# Module 7 — Instrumental Variables

Part of the Causal Inference Playbook. Course program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

- The question: does one more year of schooling raise a man's pay?
- The problem: ability may raise both a man's schooling and his pay, and the data measures ability only in part: the closest column is an IQ score, and 949 of the 3,010 men have no IQ score. So comparing men with more and less schooling mixes the effect of schooling with the effect of ability.
- The way around: an instrument. An instrument must meet two rules:
  1. The schooling rule: the instrument changes how much schooling some men get.
  2. The no-other-way rule: the instrument affects pay only through schooling. So men with and without the instrument must also be alike in ability. The data can check ability only in part, using the IQ score.
- The instrument tested here: living near a college in 1966. A nearby college is cheaper to attend, because a student can live at home.

## From the curriculum

> ## <u>Module 7 — Instrumental Variables</u>
>
> **Concepts**
> - Instrument: something that changes how much of a treatment people get, and
>   affects the outcome only through that treatment. Here, the treatment is
>   schooling, the outcome is pay, and the instrument is living near a college.
> - The schooling rule: the instrument must change how much schooling men get.
>   The data can check this rule.
> - The no-other-way rule: the instrument must affect pay only through schooling.
>   The data can look for other routes, but can't prove this rule.
> - 2SLS (two-stage least squares): two regressions that give one pay number for
>   a year of schooling. 2SLS estimate = (pay gap) ÷ (schooling gap), both between
>   men with the instrument and men without the instrument.
> - The F-statistic: how many standard errors the schooling gap is above 0,
>   squared. The usual pass mark is 10.
> - Which men the 2SLS estimate describes: only men who stay in school longer when a
>   college is close, assuming a close college never shortens schooling.
>
> **Why it matters**
> Comparing men with more and less schooling mixes the effect of schooling with
> the effect of ability, and the data measures ability only in part. An instrument
> gives a number without an ability column. The price is a rule that can't be
> proved, an interval wider than the plain regression's, and a 2SLS estimate that
> describes only some men.
>
> **Worked example**
> Data from David Card's 1995 study: 3,010 US men in 1976, with living near a
> four-year college in 1966 as the instrument. A regression of pay on
> schooling, the instrument's calculation, 2SLS with the city column in both
> regressions, the F-statistic, and a check of the no-other-way rule using region.
>
> **Resources**
> - [Causal Inference: The Mixtape, ch. 7 "Instrumental Variables"](https://mixtape.scunning.com/07-instrumental_variables)
>   (Scott Cunningham, ~120 min, verified): two-stage least squares (7.3), weak
>   instruments (7.4), and which people the number describes (7.5).
> - [The Effect, ch. 19 "Instrumental Variables"](https://theeffectbook.net/ch-InstrumentalVariables.html)
>   (Nick Huntington-Klein, ~60 min, verified): the two rules (19.1.2) and how
>   2SLS is run (19.2).
>
> **Exercise type:** redo 2SLS with a different instrument from the same data,
> living near a two-year college. Fully worked.
>
> **Interview angle:** "We can't force users to adopt feature X, but we randomly
> sent half of them an email promoting feature X. The email group spent $2 more
> per user, and 10 percentage points more of the email group used feature X. What
> does feature X add to spend?"

## The data

3,010 US men aged 24 to 34 in 1976, so aged 14 to 24 in 1966, from the data David Card used in a 1995 study of schooling and pay. Each row is one man. This section uses four of the file's columns:
- Pay: hourly pay in 1976, in dollars.
- Schooling: years of schooling, as of 1976.
- Near college: 1 if a four-year college was in the man's local area in 1966, otherwise 0. The data doesn't give a distance.
- City: 1 if the man lived in a metropolitan area in 1966, otherwise 0.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

import common

card = pd.read_stata(common.RAW / "card.dta")
# The raw pay column is in cents.
card = card.assign(pay=card.wage.astype(float) / 100, schooling=card.educ, near_college=card.nearc4.astype(int), city=card.smsa66.astype(int))

## The plain comparison

What does a regression of pay on schooling give, with no instrument? This number mixes schooling and ability, as explained above. It is the number to compare the instrument's number with.

In [2]:
together = ((card.schooling - card.schooling.mean()) * (card.pay - card.pay.mean())).mean()
spread = ((card.schooling - card.schooling.mean()) ** 2).mean()
plain = smf.ols("pay ~ schooling", card).fit().params["schooling"]
print(f"Schooling and pay together = mean of [(man's schooling - mean schooling) x (man's pay - mean pay)] (3,010 men) = {together:.2f}")
print(f"Schooling spread = mean of [(man's schooling - mean schooling)²] (3,010 men) = {spread:.2f}")
print(f"Pay per extra year of schooling, plain regression = (schooling and pay together) / (schooling spread) = {together:.2f} / {spread:.2f} = ${plain:.2f} per hour")

Schooling and pay together = mean of [(man's schooling - mean schooling) x (man's pay - mean pay)] (3,010 men) = 2.12
Schooling spread = mean of [(man's schooling - mean schooling)²] (3,010 men) = 7.16
Pay per extra year of schooling, plain regression = (schooling and pay together) / (schooling spread) = 2.12 / 7.16 = $0.30 per hour


## The instrument's calculation

What is one year of schooling worth, using only the two gaps between the near-college men and the not-near men?

- Pay gap = (mean pay, near-college men) − (mean pay, not-near men).
- Schooling gap = (mean schooling, near-college men) − (mean schooling, not-near men).
- If near college affects pay only through schooling, the schooling gap causes all of the pay gap. So to get the pay of one year, split the pay gap over the years of the schooling gap: pay per extra year = (pay gap) ÷ (schooling gap).

In [3]:
g = card.groupby("near_college").agg(men=("pay", "size"), pay=("pay", "mean"), schooling=("schooling", "mean"))
far, near = g.loc[0], g.loc[1]
pay_gap, school_gap = round(near.pay, 2) - round(far.pay, 2), round(near.schooling, 2) - round(far.schooling, 2)
print(f"Pay gap = (mean pay, near college, {near.men:,.0f} men) - (mean pay, not near college, {far.men:,.0f} men) = {near.pay:.2f} - {far.pay:.2f} = ${pay_gap:.2f} per hour")
print(f"Schooling gap = (mean schooling, near college, {near.men:,.0f} men) - (mean schooling, not near college, {far.men:,.0f} men) = {near.schooling:.2f} - {far.schooling:.2f} = {school_gap:.2f} years")
print(f"Pay per extra year of schooling, instrument = (pay gap) / (schooling gap) = {pay_gap:.2f} / {school_gap:.2f} = ${pay_gap / school_gap:.2f} per hour")

Pay gap = (mean pay, near college, 2,053 men) - (mean pay, not near college, 957 men) = 6.06 - 5.16 = $0.90 per hour
Schooling gap = (mean schooling, near college, 2,053 men) - (mean schooling, not near college, 957 men) = 13.53 - 12.70 = 0.83 years
Pay per extra year of schooling, instrument = (pay gap) / (schooling gap) = 0.90 / 0.83 = $1.08 per hour


**Observation:** the instrument's calculation gives $1.08 per hour, against the plain regression's $0.30. Removing ability should lower the plain regression's number, not raise the number. So the next section checks the no-other-way rule.

## Does near college affect pay in another way?

The $1.08 holds only if the no-other-way rule holds. The rule is in doubt if near-college men differ in something else that goes with higher pay. City is a column the data has, so city can be checked. The rule is in doubt if both of these hold:
1. Near-college men lived in a city more often.
2. Men from a city earn more, even with the same schooling. The check uses men with exactly 12 years of schooling, the most common value (992 of the 3,010 men).

In [4]:
share = card.groupby("near_college").city.mean()
print(f"{card[card.near_college == 1].city.sum():,.0f} men lived in a city in 1966, out of the {near.men:,.0f} near-college men = {share[1]:.1%}")
print(f"{card[card.near_college == 0].city.sum():,.0f} men lived in a city in 1966, out of the {far.men:,.0f} not-near-college men = {share[0]:.1%}")
print()
twelve = card[card.schooling == 12].groupby("city").pay.agg(["size", "mean"])
print(f"Mean pay, 12 years of schooling, in a city in 1966 ({twelve.loc[1, 'size']:,.0f} men) = ${twelve.loc[1, 'mean']:.2f} per hour")
print(f"Mean pay, 12 years of schooling, not in a city in 1966 ({twelve.loc[0, 'size']:,.0f} men) = ${twelve.loc[0, 'mean']:.2f} per hour")

1,641 men lived in a city in 1966, out of the 2,053 near-college men = 79.9%
314 men lived in a city in 1966, out of the 957 not-near-college men = 32.8%

Mean pay, 12 years of schooling, in a city in 1966 (634 men) = $6.04 per hour
Mean pay, 12 years of schooling, not in a city in 1966 (358 men) = $4.92 per hour


**Observation:** both hold. Near-college men lived in a city far more often, 79.9% against 32.8%. And among men with 12 years of schooling, men from a city earn more.

**Conclusion:** the no-other-way rule is in doubt, so the $1.08 can't be read as the effect of schooling yet: some of the $0.90 pay gap may come from cities, and the $1.08 would count that part as schooling.

## Does the $1.08 fall when city is held fixed?

The $1.08 may count pay that comes from having lived in a city in 1966. Holding city fixed means comparing only men with the same value in the city column. Those men can't differ in that column, so that column can't cause the pay gap between them. So this section runs the instrument's calculation twice: once among men in a city in 1966, and once among men not in a city in 1966.

Holding city fixed can change both the pay gap and the schooling gap, so the two results can land above or below $1.08.

- If both results land below $1.08, part of the $1.08 may have come from cities.
- If both results land at $1.08 or above, no part of the $1.08 may have come from cities.
- If one result lands below $1.08 and the other at $1.08 or above, the two results disagree, so this check can't say whether part of the $1.08 came from cities.

The output shows three decimals so each division can be checked by hand.

In [5]:
for city, label in [(1, "in a city in 1966"), (0, "not in a city in 1966")]:
    men = card[card.city == city]
    close, other = men[men.near_college == 1], men[men.near_college == 0]
    pay_gap = close.pay.mean() - other.pay.mean()
    school_gap = close.schooling.mean() - other.schooling.mean()
    print(f"Men {label}:")
    print(f"  Pay gap = (mean pay, near college, {len(close):,} men) - (mean pay, not near college, {len(other):,} men) = {close.pay.mean():.3f} - {other.pay.mean():.3f} = ${pay_gap:.3f} per hour")
    print(f"  Schooling gap = (mean schooling, near college, {len(close):,} men) - (mean schooling, not near college, {len(other):,} men) = {close.schooling.mean():.3f} - {other.schooling.mean():.3f} = {school_gap:.3f} years")
    print(f"  Pay per extra year of schooling, instrument = (pay gap) / (schooling gap) = {pay_gap:.3f} / {school_gap:.3f} = ${pay_gap / school_gap:.2f} per hour")
    print()

Men in a city in 1966:
  Pay gap = (mean pay, near college, 1,641 men) - (mean pay, not near college, 314 men) = 6.231 - 5.881 = $0.350 per hour
  Schooling gap = (mean schooling, near college, 1,641 men) - (mean schooling, not near college, 314 men) = 13.616 - 13.197 = 0.419 years
  Pay per extra year of schooling, instrument = (pay gap) / (schooling gap) = 0.350 / 0.419 = $0.84 per hour

Men not in a city in 1966:
  Pay gap = (mean pay, near college, 412 men) - (mean pay, not near college, 643 men) = 5.363 - 4.815 = $0.548 per hour
  Schooling gap = (mean schooling, near college, 412 men) - (mean schooling, not near college, 643 men) = 13.172 - 12.454 = 0.718 years
  Pay per extra year of schooling, instrument = (pay gap) / (schooling gap) = 0.548 / 0.718 = $0.76 per hour



**Observation:** both results, $0.84 and $0.76, are below $1.08. Both are still above the plain regression's $0.30, and the plain regression's number was expected to be the higher one, because the plain regression's number includes ability. Both results rest on fewer men than the $1.08, with as few as 314 men in one group, so each result is less exact than the $1.08.

**Hypothesis:** part of the $1.08 came from cities. This isn't proven, for two reasons: the page doesn't yet show how far $1.08, $0.84 and $0.76 could move in another sample of men, and the city column may go with other differences between the men. Later in this module, the two results are combined into one number, because the question needs one answer, and that number gets a 95% confidence interval.

## Are near-college men and not-near-college men alike in ability?

The no-other-way rule also needs near-college men and not-near-college men to be alike in ability. The closest column is the IQ score, a test score that 2,061 of the 3,010 men have. So the groups below contain only men with a score, and are smaller than the groups above. The men are split by the city column, so each IQ check sits beside the matching result above: men in a city with the $0.84, men not in a city with the $0.76.

- IQ gap = (mean IQ score, near college) − (mean IQ score, not near college).
- Variance = mean of (man's IQ score − mean IQ score)², within one group. This is the same calculation as the schooling spread above.
- The standard error is the usual size of the change in the IQ gap from one sample of men to another. Standard error = square root of [(variance ÷ men, near college) + (variance ÷ men, not near college)].
- The 95% confidence interval is the range the IQ gap could fall in if the study were rerun on another sample of men. The 1.96 is the standard multiplier for a 95% range.

We want 0 inside the interval, because the no-other-way rule needs no pay effect from an IQ gap, and with no IQ gap there is no such effect. 0 inside doesn't prove the IQ gap is exactly 0. 0 inside means this sample can't tell the IQ gap from no gap.

In [6]:
for city, label in [(1, "in a city in 1966"), (0, "not in a city in 1966")]:
    men = card[card.city == city]
    close, other = men[men.near_college == 1].IQ.dropna(), men[men.near_college == 0].IQ.dropna()
    gap = close.mean() - other.mean()
    se = (close.var() / len(close) + other.var() / len(other)) ** 0.5
    print(f"Men {label}:")
    print(f"  IQ gap = (mean IQ score, near college, {len(close):,} men) - (mean IQ score, not near college, {len(other):,} men) = {close.mean():.2f} - {other.mean():.2f} = {gap:.2f}")
    print(f"  Standard error = square root of [(variance / men, near college) + (variance / men, not near college)] = square root of [({close.var():.1f} / {len(close):,}) + ({other.var():.1f} / {len(other):,})] = {se:.2f}")
    print(f"  95% confidence interval = (IQ gap) +/- (1.96 x standard error) = {gap:.2f} +/- (1.96 x {se:.2f}) = {round(gap, 2) - 1.96 * round(se, 2):.2f} to {round(gap, 2) + 1.96 * round(se, 2):.2f}")
    print()

Men in a city in 1966:
  IQ gap = (mean IQ score, near college, 1,189 men) - (mean IQ score, not near college, 238 men) = 103.45 - 102.19 = 1.26
  Standard error = square root of [(variance / men, near college) + (variance / men, not near college)] = square root of [(236.2 / 1,189) + (237.7 / 238)] = 1.09
  95% confidence interval = (IQ gap) +/- (1.96 x standard error) = 1.26 +/- (1.96 x 1.09) = -0.88 to 3.40

Men not in a city in 1966:
  IQ gap = (mean IQ score, near college, 271 men) - (mean IQ score, not near college, 363 men) = 102.14 - 99.57 = 2.57
  Standard error = square root of [(variance / men, near college) + (variance / men, not near college)] = square root of [(227.9 / 271) + (241.2 / 363)] = 1.23
  95% confidence interval = (IQ gap) +/- (1.96 x standard error) = 2.57 +/- (1.96 x 1.23) = 0.16 to 4.98



**Observation:** among men in a city in 1966, the IQ gap is 1.26 and 0 is inside the interval, as we want, so this sample can't tell that IQ gap from no gap. Among men not in a city in 1966, the IQ gap is 2.57 and 0 is outside the interval, so near-college men in that group have higher IQ scores than not-near-college men in that group. The two intervals share most of their range, so the check doesn't show that the two IQ gaps differ. An IQ gap also doesn't say how much pay the IQ gap adds.

**Conclusion:** the no-other-way rule stays in doubt, because near-college men may have more ability, and ability may raise pay without schooling.

## What is one pay number for a year of schooling, with city held fixed?

The section "Does the $1.08 fall when city is held fixed?" gave two results, $0.84 and $0.76, and the question needs one answer. A simple average would give both results equal weight, although the two results rest on different numbers of men. Two-stage least squares (2SLS) gives one number without choosing weights by hand, using two regressions. The method drops the part of schooling that near college and city don't predict, because that part can come from ability.

- Stage 1: a regression of schooling on near college and city gives each man a predicted schooling.
- Stage 2: a regression of pay on predicted schooling and city. The regression picks the numbers that make the sum of [(man's pay) − (equation's pay)]² across the 3,010 men as small as possible. The 2SLS estimate is the amount the stage 2 equation multiplies predicted schooling by.
- How city is held fixed: stage 2 gives city its own amount, so the amount on predicted schooling doesn't include the pay difference between city values.
- How ability is kept out: stage 1 has no ability column, so two men with the same near-college and city values get the same predicted schooling, even if one man has more ability.
- When ability still gets in: if near-college men have more ability on average than not-near-college men, stage 2 counts the pay from that extra ability as pay from schooling, so the 2SLS estimate comes out too high.
- The standard error of the 2SLS estimate is the usual size of the change in the 2SLS estimate from one sample of men to another. Stage 2 is run on predicted schooling, so stage 2's own standard error is wrong: the error in a man's pay must be measured from his real schooling. Corrected standard error = square root of [(mean squared pay error) ÷ (sum of squared differences in predicted schooling)], in dollars per hour. With the city column, each difference is from the mean predicted schooling of men with the same city value.
- 95% confidence interval = (2SLS estimate) ± (1.96 × standard error).

The method, 2SLS (stage 1 followed by stage 2), runs twice:

1. With no city column in either regression. Predicted schooling then has only two values, 13.53 (mean schooling, near-college men) and 12.70 (mean schooling, not-near-college men). Predicted schooling takes those two values because stage 1 then predicts each group's mean schooling. The two points are (12.70, mean pay 5.16) and (13.53, mean pay 6.06). A straight line through two points has a slope of (pay gap) ÷ (schooling gap). So this run must give the $1.08, which checks the method. The stage 2 equation's printed amounts are rounded, so that equation matches those two points only roughly.
2. With the city column in both regressions. The 2SLS estimate from this run is the one number we wanted.

We want 0 outside each interval, because here we are looking for an effect of schooling on pay. The IQ check wanted 0 inside, because that check was looking for no gap.

In [7]:
def two_stage(columns):
    extra = "".join(f" + {name}" for name in columns)
    stage_1 = smf.ols(f"schooling ~ near_college{extra}", card).fit()
    with_predicted = card.assign(predicted=stage_1.fittedvalues)
    stage_2 = smf.ols(f"pay ~ predicted{extra}", with_predicted).fit()
    pay_from_real_schooling = stage_2.params["Intercept"] + stage_2.params["predicted"] * card.schooling.astype(float)
    if columns:
        pay_from_real_schooling = pay_from_real_schooling + stage_2.params["city"] * card.city
    pay_error = card.pay - pay_from_real_schooling
    mean_squared_error = (pay_error ** 2).sum() / (len(card) - len(stage_2.params))
    if columns:
        differences = with_predicted.predicted - with_predicted.groupby("city").predicted.transform("mean")
    else:
        differences = with_predicted.predicted - with_predicted.predicted.mean()
    return stage_1.params, stage_2.params, mean_squared_error, (differences ** 2).sum()

for columns, label in [([], "no city column"), (["city"], "city column in both regressions")]:
    first, second, mean_squared_error, sum_squared = two_stage(columns)
    estimate = second["predicted"]
    se = (mean_squared_error / sum_squared) ** 0.5
    city_1 = f" + ({first['city']:.2f} x city)" if columns else ""
    city_2 = f" + ({second['city']:.2f} x city)" if columns else ""
    print(f"2SLS, {label} (3,010 men):")
    print(f"  Stage 1: predicted schooling = {first['Intercept']:.2f} + ({first['near_college']:.2f} x near college){city_1}")
    print(f"  Stage 2: pay = {second['Intercept']:.2f} + ({estimate:.2f} x predicted schooling){city_2}")
    print(f"  Pay per extra year of schooling, 2SLS estimate = ${estimate:.2f} per hour")
    print(f"  Standard error of the 2SLS estimate = square root of [(mean squared pay error) / (sum of squared differences in predicted schooling)] = square root of [{mean_squared_error:.2f} / {sum_squared:.1f}] = {se:.2f}")
    print(f"  95% confidence interval = (2SLS estimate) +/- (1.96 x standard error) = {estimate:.2f} +/- (1.96 x {se:.2f}) = ${round(estimate, 2) - 1.96 * round(se, 2):.2f} to ${round(estimate, 2) + 1.96 * round(se, 2):.2f} per hour")
    print()

2SLS, no city column (3,010 men):
  Stage 1: predicted schooling = 12.70 + (0.83 x near college)
  Stage 2: pay = -8.49 + (1.08 x predicted schooling)
  Pay per extra year of schooling, 2SLS estimate = $1.08 per hour
  Standard error of the 2SLS estimate = square root of [(mean squared pay error) / (sum of squared differences in predicted schooling)] = square root of [10.64 / 448.6] = 0.15
  95% confidence interval = (2SLS estimate) +/- (1.96 x standard error) = 1.08 +/- (1.96 x 0.15) = $0.79 to $1.37 per hour

2SLS, city column in both regressions (3,010 men):
  Stage 1: predicted schooling = 12.51 + (0.56 x near college) + (0.56 x city)
  Stage 2: pay = -5.04 + (0.79 x predicted schooling) + (0.50 x city)
  Pay per extra year of schooling, 2SLS estimate = $0.79 per hour
  Standard error of the 2SLS estimate = square root of [(mean squared pay error) / (sum of squared differences in predicted schooling)] = square root of [7.98 / 164.2] = 0.22
  95% confidence interval = (2SLS estimate

**Observation:** with no city column, the 2SLS estimate equals the $1.08, as it must. With the city column, the 2SLS estimate is $0.79, which combines the $0.84 and the $0.76 into one number and is below $1.08. 0 is outside both intervals, as we want. $1.08 is inside the interval of $0.36 to $1.22, so the hypothesis that part of the $1.08 came from cities stays unproven. The 2SLS estimate of $0.79 is still higher than the plain regression's $0.30, which doesn't hold city fixed, although removing ability should make an estimate lower than the plain regression's. A later section explains why the 2SLS estimate can be higher.

**Conclusion:** schooling raises pay if the no-other-way rule holds, because 0 is outside the interval of the run with the city column. That rule is still in doubt.

## Is the near-college schooling gap large enough for a reliable 2SLS estimate?

The schooling rule says that near-college men must have more schooling than not-near-college men, with city held fixed. With city held fixed, stage 1 measures that schooling gap with one number, the near-college coefficient: the number that stage 1 multiplies the near-college column by. Stage 1 prints the near-college coefficient as about 0.56 years (0.565 years at three decimals).

We check whether about 0.56 years is large enough for a reliable 2SLS estimate of $0.79 per hour. Here, a reliable 2SLS estimate means a 2SLS estimate that isn't pushed far off by chance differences between samples of men. The test comes at the end of the section.

The size of the near-college coefficient matters because the 2SLS estimate of $0.79 per hour = (weighted pay gap) ÷ (near-college coefficient). The weighted pay gap is how much more near-college men earn per hour than not-near-college men, with city held fixed. The near-college coefficient is how many more years of schooling near-college men have, with city held fixed. Below, we show why stage 2 gives that division, then calculate the weighted pay gap and the near-college coefficient.

- Within each city group, predicted schooling has two values, one near-college coefficient apart.
- Stage 2 fits pay against predicted schooling. Stage 2's city coefficient, the number stage 2 multiplies the city column by, holds the pay difference between the city groups, so the number stage 2 multiplies predicted schooling by doesn't include that pay difference.
- The slope of the stage 2 line is the 2SLS estimate of $0.79 per hour. Slope = (pay gap between the two values) ÷ (distance between the two values).

The pay gap and the distance are each one number for two city groups, so the pay gap and the distance are each a weighted mean across the two city groups. A city group's weight is large when the group has many near-college men and many not-near-college men, because a gap between two large sets of men is more exact. The regression sets these weights, and we calculate the weights here to see them. The appendix shows the same weights in stage 1.

In [8]:
stage_1 = smf.ols("schooling ~ near_college + city", card).fit()
pay_gaps, school_gaps, weights = [], [], []
print("Weight = (men near college x men not near college) / (all men in the city group)")
for city, label in [(1, "men in a city"), (0, "men not in a city")]:
    men = card[card.city == city]
    close, other = men[men.near_college == 1], men[men.near_college == 0]
    pay_gaps.append(round(close.pay.mean() - other.pay.mean(), 3))
    school_gaps.append(round(close.schooling.mean() - other.schooling.mean(), 3))
    weights.append(round(len(close) * len(other) / (len(close) + len(other)), 1))
    print(f"  Weight, {label} = ({len(close):,} x {len(other):,}) / {len(close) + len(other):,} = {weights[-1]:.1f}")
total_weight = round(sum(weights), 1)
print(f"  Sum of squared differences in near college = sum of the two weights = {weights[0]:.1f} + {weights[1]:.1f} = {total_weight:.1f}")

Weight = (men near college x men not near college) / (all men in the city group)
  Weight, men in a city = (1,641 x 314) / 1,955 = 263.6
  Weight, men not in a city = (412 x 643) / 1,055 = 251.1
  Sum of squared differences in near college = sum of the two weights = 263.6 + 251.1 = 514.7


Here are the two values of predicted schooling in each city group, with the pay at each:

In [9]:
start, near_number, city_number = (round(stage_1.params[name], 2) for name in ["Intercept", "near_college", "city"])
print(f"{'Men in the group':<18}{'Near college or not':<24}{'Predicted schooling':>22}{'Mean pay per hour':>20}")
for city, label in [(1, "In a city"), (0, "Not in a city")]:
    men = card[card.city == city]
    close_pay, other_pay = round(men[men.near_college == 1].pay.mean(), 3), round(men[men.near_college == 0].pay.mean(), 3)
    close_school = start + near_number + city_number * city
    other_school = start + city_number * city
    print(f"{label:<18}{'Near college':<24}{close_school:>16.2f} years{'$' + format(close_pay, '.3f'):>20}")
    print(f"{label:<18}{'Not near college':<24}{other_school:>16.2f} years{'$' + format(other_pay, '.3f'):>20}")
    print(f"{label:<18}{'Distance and pay gap':<24}{close_school - other_school:>16.2f} years{'$' + format(close_pay - other_pay, '.3f'):>20}")

Men in the group  Near college or not        Predicted schooling   Mean pay per hour
In a city         Near college                       13.63 years              $6.231
In a city         Not near college                   13.07 years              $5.881
In a city         Distance and pay gap                0.56 years              $0.350
Not in a city     Near college                       13.07 years              $5.363
Not in a city     Not near college                   12.51 years              $4.815
Not in a city     Distance and pay gap                0.56 years              $0.548


The distance is the near-college coefficient, the same in both city groups. The pay gaps of $0.350 and $0.548 differ, because the real schooling gaps differ: 0.419 years in a city and 0.718 years not in a city, from the section "Does the $1.08 fall when city is held fixed?". Stage 2 gives one slope for both city groups, so we need one pay gap. The next output combines the pay gaps of $0.350 and $0.548 into one weighted pay gap.

In [10]:
weighted_pay_gap = (pay_gaps[0] * weights[0] + pay_gaps[1] * weights[1]) / total_weight
coefficient = (school_gaps[0] * weights[0] + school_gaps[1] * weights[1]) / total_weight
print("Weighted pay gap, 3,010 men:")
print("  = [(pay gap in a city x weight in a city) + (pay gap not in a city x weight not in a city)] / (sum of the two weights)")
print(f"  = [({pay_gaps[0]:.3f} x {weights[0]:.1f}) + ({pay_gaps[1]:.3f} x {weights[1]:.1f})] / {total_weight:.1f} = ${weighted_pay_gap:.3f} per hour")
print()
print("Near-college coefficient, 3,010 men:")
print("  = [(schooling gap in a city x weight in a city) + (schooling gap not in a city x weight not in a city)] / (sum of the two weights)")
print(f"  = [({school_gaps[0]:.3f} x {weights[0]:.1f}) + ({school_gaps[1]:.3f} x {weights[1]:.1f})] / {total_weight:.1f} = {coefficient:.3f} years")

Weighted pay gap, 3,010 men:
  = [(pay gap in a city x weight in a city) + (pay gap not in a city x weight not in a city)] / (sum of the two weights)
  = [(0.350 x 263.6) + (0.548 x 251.1)] / 514.7 = $0.447 per hour

Near-college coefficient, 3,010 men:
  = [(schooling gap in a city x weight in a city) + (schooling gap not in a city x weight not in a city)] / (sum of the two weights)
  = [(0.419 x 263.6) + (0.718 x 251.1)] / 514.7 = 0.565 years


We said that the 2SLS estimate = (weighted pay gap) ÷ (near-college coefficient). Stage 2 uses the near-college coefficient inside the regression, where we can't see it. The division uses the same near-college coefficient where we can see it. To test that stage 2 and the division give the same number, we divide and compare the result with the $0.79 per hour that stage 2 printed.

In [11]:
print("2SLS estimate, 3,010 men:")
print("  = (weighted pay gap) / (near-college coefficient)")
print(f"  = {weighted_pay_gap:.3f} / {coefficient:.3f} = ${round(weighted_pay_gap, 3) / round(coefficient, 3):.2f} per hour")

2SLS estimate, 3,010 men:
  = (weighted pay gap) / (near-college coefficient)
  = 0.447 / 0.565 = $0.79 per hour


The near-college coefficient is the bottom of the division, and a small bottom number is a problem. The weighted pay gap changes from one sample of men to another, and dividing by a small number makes those changes large. The numbers below are a chosen example, not data.

In [12]:
print("Chosen example, a weighted pay gap of $0.10 per hour in one sample of men and $0.20 per hour in another sample of men:")
print()
for bottom in [0.01, 0.50]:
    print(f"  Near-college coefficient = {bottom:.2f} years:")
    print("    2SLS estimate = (weighted pay gap) / (near-college coefficient)")
    print(f"    First sample = 0.10 / {bottom:.2f} = ${0.10 / bottom:.2f} per hour")
    print(f"    Second sample = 0.20 / {bottom:.2f} = ${0.20 / bottom:.2f} per hour")
    print(f"    Change between samples = {0.20 / bottom:.2f} - {0.10 / bottom:.2f} = ${0.20 / bottom - 0.10 / bottom:.2f} per hour")
    print()

Chosen example, a weighted pay gap of $0.10 per hour in one sample of men and $0.20 per hour in another sample of men:

  Near-college coefficient = 0.01 years:
    2SLS estimate = (weighted pay gap) / (near-college coefficient)
    First sample = 0.10 / 0.01 = $10.00 per hour
    Second sample = 0.20 / 0.01 = $20.00 per hour
    Change between samples = 20.00 - 10.00 = $10.00 per hour

  Near-college coefficient = 0.50 years:
    2SLS estimate = (weighted pay gap) / (near-college coefficient)
    First sample = 0.10 / 0.50 = $0.20 per hour
    Second sample = 0.20 / 0.50 = $0.40 per hour
    Change between samples = 0.40 - 0.20 = $0.20 per hour



Is the near-college coefficient of 0.565 years small compared with the usual change in the near-college coefficient from one sample of men to another? That usual change is the standard error of the near-college coefficient. A near-college coefficient about as large as the standard error could be close to 0 in another sample of men.

The near-college coefficient is a schooling gap between two groups of men, so the near-college coefficient changes more from one sample of men to another when men's schooling is more spread out. The standard error measures that spread by how far each man's schooling is from the man's predicted schooling.

- Schooling error = (man's schooling) − (man's predicted schooling from stage 1, with the city column), because the 0.565 years comes from stage 1 with the city column.
- Mean squared schooling error = (sum of squared schooling errors) ÷ (3,010 men − 3). The 3 counts the three numbers in the stage 1 equation, 12.51, 0.56 and 0.56: stage 1 chose those three numbers to make the schooling errors small, so dividing by all 3,010 men would make the mean too small.

In [13]:
schooling_error = card.schooling.astype(float) - stage_1.fittedvalues
sum_squared = (schooling_error ** 2).sum()
mean_squared_error = sum_squared / (len(card) - len(stage_1.params))
se = (mean_squared_error / total_weight) ** 0.5
print("Mean squared schooling error, 3,010 men:")
print(f"  = (sum of squared schooling errors) / (3,010 men - 3) = {sum_squared:,.1f} / 3,007 = {mean_squared_error:.2f}")

Mean squared schooling error, 3,010 men:
  = (sum of squared schooling errors) / (3,010 men - 3) = 20,943.6 / 3,007 = 6.96


The standard error of the near-college coefficient divides the mean squared schooling error by the sum of squared differences in near college, because a larger sum of squared differences in near college means larger sets of men and a more exact gap. The square root returns squared years to years.

In [14]:
print("Standard error of the near-college coefficient, 3,010 men:")
print("  = square root of [(mean squared schooling error) / (sum of squared differences in near college)]")
print(f"  = square root of [{mean_squared_error:.2f} / {total_weight:.1f}] = {se:.4f} years")

Standard error of the near-college coefficient, 3,010 men:
  = square root of [(mean squared schooling error) / (sum of squared differences in near college)]
  = square root of [6.96 / 514.7] = 0.1163 years


Now we can answer the question: is the near-college coefficient of 0.565 years small compared with the standard error of 0.1163 years? The comparison is called the F-statistic: how many standard errors the near-college coefficient is above 0, squared. "F" is the name of the test, and the square is there because the test's formula works with squared differences.

The pass mark for the F-statistic is 10, from a 1997 study by Staiger and Stock. Below 10, the 2SLS estimate can be far from the true effect, so we want an F-statistic above 10.

In [15]:
print("F-statistic, 3,010 men:")
print("  = [(near-college coefficient) / (standard error of the near-college coefficient)]²")
print(f"  = ({coefficient:.3f} / {se:.4f})² = {(round(coefficient, 3) / round(se, 4)) ** 2:.1f}")

F-statistic, 3,010 men:
  = [(near-college coefficient) / (standard error of the near-college coefficient)]²
  = (0.565 / 0.1163)² = 23.6


**Observation:** the F-statistic of 23.6 is above 10, as we want. The F-statistic checks only the size of the near-college coefficient. The F-statistic doesn't check whether near college raises pay in a way other than schooling, so the F-statistic doesn't test the no-other-way rule.

**Conclusion:** the near-college schooling gap is large enough for a reliable 2SLS estimate. The no-other-way rule, that near college raises pay only through schooling, is still in doubt, so we can't yet say that schooling raises pay.

## Why is the 2SLS estimate of $0.79 per hour higher than the plain regression's $0.27 per hour?

The plain regression counts the pay from ability as pay from schooling, so removing ability should give a number lower than the plain regression's. The 2SLS estimate is higher instead. We look at three of the possible explanations, one at a time: chance, different men, and a broken no-other-way rule.

**Explanation 1, chance:** the $0.79 per hour comes from one sample of men, so the true number could be lower. The 95% confidence interval from the section "What is one pay number for a year of schooling, with city held fixed?" gives the lowest number that chance plausibly allows. If the interval includes the plain regression with the city column, chance alone could explain the difference. If the interval doesn't include the plain regression with the city column, chance alone is unlikely.

The plain regression so far has no city column, and that plain regression gave $0.30 per hour. The 95% confidence interval of the 2SLS estimate comes from the run with the city column, so that interval holds city fixed. Comparing a number that holds city fixed with a number that doesn't would mix two differences: the difference between the methods, and the difference from holding city fixed. To keep only the difference between the methods, we now run the plain regression with the city column too. Each man is compared with men who have his city value, so pay that goes with living in a city doesn't count as pay from schooling.

In [16]:
schooling_gap = card.schooling.astype(float) - card.groupby("city").schooling.transform(lambda s: s.astype(float).mean())
pay_gap = card.pay - card.groupby("city").pay.transform("mean")
together = (schooling_gap * pay_gap).mean()
spread = (schooling_gap ** 2).mean()
plain_city = smf.ols("pay ~ schooling + city", card).fit().params["schooling"]
print(f"Schooling and pay together, city column = mean of [(man's schooling - mean schooling of men with his city value) x (man's pay - mean pay of men with his city value)] (3,010 men) = {together:.2f}")
print(f"Schooling spread, city column = mean of [(man's schooling - mean schooling of men with his city value)²] (3,010 men) = {spread:.2f}")
print(f"Pay per extra year of schooling, plain regression with the city column = (schooling and pay together, city column) / (schooling spread, city column) = {together:.2f} / {spread:.2f} = ${plain_city:.2f} per hour")

Schooling and pay together, city column = mean of [(man's schooling - mean schooling of men with his city value) x (man's pay - mean pay of men with his city value)] (3,010 men) = 1.91
Schooling spread, city column = mean of [(man's schooling - mean schooling of men with his city value)²] (3,010 men) = 7.01
Pay per extra year of schooling, plain regression with the city column = (schooling and pay together, city column) / (schooling spread, city column) = 1.91 / 7.01 = $0.27 per hour


In [17]:
print("95% confidence interval of the 2SLS estimate, 3,010 men = $0.36 to $1.22 per hour")
with_city = smf.ols("pay ~ schooling + city", card).fit().params["schooling"]
print("Plain regression, no city column, 3,010 men = $0.30 per hour")
print(f"Plain regression, city column, 3,010 men = ${with_city:.2f} per hour")
print(f"Distance from the plain regression with the city column to the interval's lower end = 0.36 - {with_city:.2f} = ${0.36 - round(with_city, 2):.2f} per hour")

95% confidence interval of the 2SLS estimate, 3,010 men = $0.36 to $1.22 per hour


Plain regression, no city column, 3,010 men = $0.30 per hour
Plain regression, city column, 3,010 men = $0.27 per hour
Distance from the plain regression with the city column to the interval's lower end = 0.36 - 0.27 = $0.09 per hour


The interval holds city fixed, so the fair comparison is the plain regression with the city column. The $0.27 per hour is below the interval, so chance alone is an unlikely explanation for the 2SLS estimate of $0.79 per hour being higher than the plain regression's $0.27 per hour, although the distance of $0.09 per hour between the $0.27 per hour and the interval's lower end of $0.36 per hour is small against the 2SLS estimate's standard error of $0.22 per hour.

**Explanation 2: the two numbers measure what a year of school pays for different men.**

- The plain regression's $0.27 uses every difference in schooling among the 3,010 men with the same city value, so the $0.27 is about all 3,010 men.
- The 2SLS estimate's $0.79 uses only the difference in schooling between near-college men and not-near-college men with the same city value.
- The difference comes only from men who stay in school longer when a college is close, assuming a close college never shortens schooling. The $0.79 is about those men only.
- We suspect a year of school pays those men more than the average man, which would put the $0.79 above the $0.27.
- Testing this needs a list of those men. Each man grew up in one place, so the data never shows his schooling in the other place, and no such list exists.

**Conclusion:** Explanation 2 is possible, but this data can't test it.

**Explanation 3, a broken no-other-way rule:** if near-college men earn more for a reason other than schooling, for example more ability, the 2SLS estimate, (weighted pay gap) ÷ (near-college coefficient), counts that pay as pay from schooling. How large would the pay gap from other causes have to be to bring the 2SLS estimate down from $0.79 per hour per year of schooling to the plain regression with the city column, $0.27 per hour per year of schooling? At $0.27 per hour for a year of schooling, schooling explains only part of the weighted pay gap of $0.447 per hour between near-college men and not-near-college men, and the pay gap from other causes is the rest. The weighted pay gap of $0.447 per hour and the near-college coefficient of 0.565 years come from the section "Is the near-college schooling gap large enough for a reliable 2SLS estimate?".

In [18]:
added_pay = round(0.27 * 0.565, 4)
print("What extra schooling would add, if a year of schooling pays $0.27 per hour:")
print("  Near-college men have 0.565 more years of schooling.")
print(f"  Added pay = 0.565 years x $0.27 = ${added_pay:.4f} per hour")
print()
print("What near-college men really earn more:")
print("  Weighted pay gap = $0.447 per hour")
print()
print("What extra schooling doesn't explain:")
print(f"  Left over = $0.447 - ${added_pay:.4f} = ${0.447 - added_pay:.4f} per hour")
print("  Something other than schooling would have to explain this part.")

What extra schooling would add, if a year of schooling pays $0.27 per hour:
  Near-college men have 0.565 more years of schooling.
  Added pay = 0.565 years x $0.27 = $0.1525 per hour

What near-college men really earn more:
  Weighted pay gap = $0.447 per hour

What extra schooling doesn't explain:
  Left over = $0.447 - $0.1525 = $0.2945 per hour
  Something other than schooling would have to explain this part.


**Observation:** the plain regression with the city column is $0.09 per hour below the interval of the 2SLS estimate. A pay gap from other causes of $0.2945 per hour, out of the weighted pay gap of $0.447 per hour, would bring the 2SLS estimate down to $0.27 per hour.

**Hypothesis:** the 2SLS estimate is higher for one or more of these reasons, or another: (1) chance, which is unlikely, (2) men whose schooling a nearby college changed gaining more from a year of schooling, (3) near-college men earning more for a reason other than schooling. A measure of ability for all 3,010 men would help test the third reason.

**Conclusion:** the data can't say whether different men or a broken no-other-way rule makes the 2SLS estimate higher than the plain regression.

## Does living near a college raise pay only through schooling?

The 2SLS estimate is correct only if the no-other-way rule holds: living near a college raises pay only through schooling. No calculation can prove that rule, because we can't see every other route to higher pay. We can only look for other routes in the columns we have.

One route we can see is region. Pay in 1976 was lower in the South, so if near-college men lived in the South less often, part of the near-college pay gap comes from region and not from schooling. We use a new column here. South: 1 if the man lived in the South in 1966, otherwise 0.

In [19]:
card = card.assign(south=card.south66.astype(int))
for near, label in [(1, "Near-college men"), (0, "Not-near-college men")]:
    men = card[card.near_college == near]
    print(f"{label}:")
    print(f"  Men in the South = {men.south.sum():,} of {len(men):,} = {men.south.mean():.1%}")
    print()
print("Mean pay per hour:")
for south, label in [(1, "Men in the South"), (0, "Men not in the South")]:
    men = card[card.south == south]
    print(f"  {label} ({len(men):,} men) = ${men.pay.mean():.2f}")

Near-college men:
  Men in the South = 675 of 2,053 = 32.9%

Not-near-college men:
  Men in the South = 572 of 957 = 59.8%

Mean pay per hour:
  Men in the South (1,247 men) = $4.99


  Men not in the South (1,763 men) = $6.33


Near-college men lived in the South less often, and men in the South earned less. Region is a real other route, so we hold region fixed the same way we held city fixed: we add the South column to both stages of 2SLS. We want to see whether the 2SLS estimate falls, as we suspect, and if so by how much, and whether the F-statistic stays above 10.

In [20]:
def south_run():
    stage_1 = smf.ols("schooling ~ near_college + city + south", card).fit()
    with_predicted = card.assign(predicted=stage_1.fittedvalues)
    stage_2 = smf.ols("pay ~ predicted + city + south", with_predicted).fit()
    pay_error = card.pay - (stage_2.params["Intercept"] + stage_2.params["predicted"] * card.schooling.astype(float) + stage_2.params["city"] * card.city + stage_2.params["south"] * card.south)
    mean_squared_error = (pay_error ** 2).sum() / (len(card) - len(stage_2.params))
    differences = smf.ols("predicted ~ city + south", with_predicted).fit().resid
    return stage_1, stage_2, (mean_squared_error / (differences ** 2).sum()) ** 0.5

stage_1, stage_2, se = south_run()
estimate = stage_2.params["predicted"]
print("2SLS with the city and South columns, 3,010 men:")
print()
print(f"  Near-college coefficient = {stage_1.params['near_college']:.3f} years")
print(f"  F-statistic = {stage_1.tvalues['near_college'] ** 2:.1f}")
print()
print(f"  Pay per extra year of schooling, 2SLS estimate = ${estimate:.2f} per hour")
print(f"  Standard error of the 2SLS estimate = {se:.2f}")
print(f"  95% confidence interval = {estimate:.2f} +/- (1.96 x {se:.2f}) = ${round(estimate, 2) - 1.96 * round(se, 2):.2f} to ${round(estimate, 2) + 1.96 * round(se, 2):.2f} per hour")

2SLS with the city and South columns, 3,010 men:

  Near-college coefficient = 0.377 years
  F-statistic = 10.6

  Pay per extra year of schooling, 2SLS estimate = $0.66 per hour
  Standard error of the 2SLS estimate = 0.32
  95% confidence interval = 0.66 +/- (1.96 x 0.32) = $0.03 to $1.29 per hour


**Observation:** near-college men lived in the South less often than not-near-college men, and men in the South earned less. Adding the South column lowers the 2SLS estimate from $0.79 to $0.66 per hour. The F-statistic of 10.6 is just above the pass mark of 10. The interval of $0.03 to $1.29 per hour is wide, and 0 is just outside the interval, as we want, because 0 inside the interval would mean a year of schooling may add no pay.

**Hypothesis:** other routes we can't see may lower the 2SLS estimate further, for example better local jobs near colleges.

**Conclusion:** the no-other-way rule doesn't look safe with city alone, because region looks like another route to higher pay. Holding region fixed doesn't bring the 2SLS estimate down to the plain regression ($0.27 per hour), although the remaining difference could now be chance, because the interval got wider ($0.36 to $1.22 per hour before, $0.03 to $1.29 per hour now), so the plain regression's $0.27 per hour is now inside the interval. The answer to the module's question is that one more year of schooling probably raises a man's pay, but this data can't say by exactly how much.

## Exercise — 2SLS with the two-year-college column

The file has another column that could be an instrument. Near two-year college: 1 if a two-year college was in the man's local area in 1966, otherwise 0 (column `nearc2`, 1,327 of the 3,010 men). Redo the module's calculation with this column in place of near college.

Write your answers first, then read the answers below.

### Questions

1. With no city column, what is pay per extra year of schooling, using (pay gap) ÷ (schooling gap)? This is the module's $1.08 calculation, with near two-year college in place of near college. The plain regression's $0.30 per hour doesn't use near college, so swapping the column doesn't change it.
2. With the city column in stage 1, what is the near-two-year-college coefficient? What is the F-statistic? Does the F-statistic pass the mark of 10?
3. With the city column in both stages, what are the 2SLS estimate and the 95% confidence interval? What can you conclude about pay from schooling?

### Answers

**Answer 1.**

- Pay gap = (mean pay, near a two-year college, 1,327 men) − (mean pay, not near a two-year college, 1,683 men) = 6.115 − 5.503 = $0.612 per hour.
- Schooling gap = (mean schooling, near a two-year college) − (mean schooling, not near a two-year college) = 13.406 − 13.151 = 0.255 years.
- Pay per extra year of schooling = (pay gap) ÷ (schooling gap) = 0.612 ÷ 0.255 = $2.40 per hour.
- Two things could make the schooling gap of 0.255 years: the two-year college, or city, if the two groups of men lived in a city at different rates. Answer 2 tests which one, by holding city fixed:
    - Men in a city out of all men near a two-year college = 1,023 ÷ 1,327 = 77.1%
    - Men in a city out of all men not near a two-year college = 932 ÷ 1,683 = 55.4%

**Answer 2.**

- Stage 1: predicted schooling = 12.71 + (0.083 x near two-year college) + (0.79 x city).
- Near-two-year-college coefficient, from stage 1 = 0.083 years.
- With city held fixed, the near-two-year-college coefficient is the weighted average of the schooling gaps inside the two city groups:
    - Men in a city, schooling gap = (mean schooling, near a two-year college) − (mean schooling, not near a two-year college) = 13.6794 − 13.4056 = 0.2738 years
    - Men not in a city, schooling gap = (mean schooling, near a two-year college) − (mean schooling, not near a two-year college) = 12.4868 − 12.8349 = (−0.3481) years
    - Near-two-year-college coefficient = [(men in a city, schooling gap × weight in a city) + (men not in a city, schooling gap × weight not in a city)] ÷ (sum of the two weights) = [(0.2738 × 487.7) + (−0.3481 × 216.4)] ÷ 704.1 = 0.0827 years
- Weight, men in a city = (men near a two-year college × men not near a two-year college) ÷ (all men in a city) = (1,023 × 932) ÷ 1,955 = 487.7
- Weight, men not in a city = (men near a two-year college × men not near a two-year college) ÷ (all men not in a city) = (304 × 751) ÷ 1,055 = 216.4
- Sum of squared differences in near two-year college = sum of the two weights = 487.7 + 216.4 = 704.1
- Standard error of the near-two-year-college coefficient = square root of [(mean squared schooling error) ÷ (sum of squared differences in near two-year college)] = square root of (7.02 ÷ 704.1) = 0.0998 years.
- F-statistic = [(near-two-year-college coefficient) ÷ (standard error)]² = (0.083 ÷ 0.0998)² = 0.69.
- The F-statistic of 0.69 is far below the pass mark of 10, so this column fails the schooling rule once city is held fixed.

**Answer 3.**

- Each stage picks the numbers that make the sum of squared errors across the 3,010 men as small as possible.
    - In stage 1, the error is (man's schooling) − (equation's schooling).
    - In stage 2, the error is (man's pay) − (equation's pay), using predicted schooling in place of real schooling.
- Stage 1: predicted schooling = 12.71 + (0.083 x near two-year college) + (0.79 x city) (the same stage 1 as in answer 2).
- Stage 2: pay = −54.04 + (4.64 x predicted schooling) − (2.63 x city).
- 2SLS estimate, from stage 2:
    - Pay gap, men in a city = (mean pay, near a two-year college) − (mean pay, not near a two-year college) = $0.484 per hour
    - Pay gap, men not in a city = (mean pay, near a two-year college) − (mean pay, not near a two-year college) = $0.157 per hour
    - Weighted pay gap = [(pay gap in a city × weight in a city) + (pay gap not in a city × weight not in a city)] ÷ (sum of the two weights) = [(0.484 × 487.7) + (0.157 × 216.4)] ÷ 704.1 = $0.3835 per hour
    - 2SLS estimate = (weighted pay gap) ÷ (near-two-year-college coefficient) = 0.3835 ÷ 0.0827 = $4.64 per hour, the same as stage 2 (0.0827 is the near-two-year-college coefficient of 0.083 at four decimals).
- Mean squared pay error = (sum of squared pay errors) ÷ (3,010 men − 3) = 420,649.2 ÷ 3,007 = 139.89
- Sum of squared differences in predicted schooling = (near-two-year-college coefficient)² × (sum of squared differences in near two-year college) = 0.0827² × 704.1 = 4.81
- Standard error of the 2SLS estimate = square root of [(mean squared pay error) ÷ (sum of squared differences in predicted schooling)] = square root of (139.89 ÷ 4.81) = 5.39.
- 95% confidence interval = (2SLS estimate) ± (1.96 × standard error) = 4.64 ± (1.96 × 5.39) = (−$5.92) to $15.20 per hour.
- We want 0 outside the interval, and 0 is inside, so this column can't show that schooling raises pay.

**What the answers show:** the $2.40 per hour in answer 1 looks like a result, but the schooling gap of 0.255 years mostly came from cities. Near-two-year-college men lived in a city more often, 77.1% against 55.4% (answer 1), and holding city fixed cut the schooling gap from 0.255 years to the near-two-year-college coefficient of 0.083 years (answer 2). The 0.083 years is the weighted average of the schooling gaps of 0.2738 years and (−0.3481 years) in the two city groups. The interval of (−$5.92) to $15.20 per hour shows the result: a two-year college is a poor instrument here, and the F-statistic shows that before anyone trusts the number.

## Interview angle

> "We can't force users to adopt feature X, but we randomly sent half of them an email promoting feature X. The email group spent $2 more per user, and 10 percentage points more of the email group used feature X. What does feature X add to spend?"

A strong answer: the email is an instrument, because random sending makes the two groups alike. Feature X's effect = (spend gap) ÷ (usage gap) = $2 ÷ 0.10 = $20 per user who adopted feature X because of the email. Three checks come with that number:

- Is the usage gap large enough? Calculating the F-statistic needs the standard error of the usage gap.
- Did the email raise spend only through feature X? If the email also reminded users to buy, part of the $2 isn't from feature X, and the $20 is too high. The data can't prove the no-other-way rule.
- Who does the $20 describe? Only users who adopted feature X because of the email, not users who would adopt feature X anyway, assuming the email never stops anyone from adopting feature X.

## Key takeaways

- An instrument has to pass two rules: the instrument changes the treatment, and the instrument affects the outcome only through the treatment.
- 2SLS estimate = (weighted pay gap) ÷ (near-college coefficient). With city held fixed, $0.447 per hour ÷ 0.565 years = $0.79 per hour.
- The F-statistic checks the schooling rule. Near college passed at 23.6 with the city column and at 10.6 with the city and South columns. A two-year college failed at 0.69.
- No calculation proves the no-other-way rule. Holding region fixed lowered the 2SLS estimate from $0.79 to $0.66 per hour, so region looks like another route to higher pay.
- Holding region fixed cost precision: the interval widened from $0.36 to $1.22 per hour to $0.03 to $1.29 per hour.
- The 2SLS estimate describes only men who stay in school longer when a college is close, so the 2SLS estimate and the plain regression may not describe the same men.
- One more year of schooling probably raises a man's pay, but this data can't say by exactly how much.

## Appendix: where stage 1's numbers come from

The section "What is one pay number for a year of schooling, with city held fixed?" prints this stage 1 equation for the run with the city column: predicted schooling = 12.51 + (0.56 x near college) + (0.56 x city). We calculate those three numbers by hand, to show that the regression's numbers are ordinary means and gaps. The two 0.56 numbers come first, because the 12.51 is calculated from the two 0.56 numbers.

**Step 1, the near-college coefficient, the first 0.56 (0.565 with three decimals).** The near-college coefficient is one number for two city groups, with two different schooling gaps, so the near-college coefficient is a weighted mean of the two schooling gaps.

A city group's weight measures how much the near-college column varies inside the group: the sum of squared differences between each man's near-college value and the group's mean near-college value. Each near-college man differs from that mean by the share of not-near-college men, and each not-near-college man differs by the share of near-college men. Adding the squares of those differences gives the weight formula in the output.

In [21]:
def cell(near, city):
    return card[(card.near_college == near) & (card.city == city)]

def weight(a, b):
    return a * b / (a + b)

gaps, weights = [], []
print("Weight = (men near college x men not near college) / (all men in the group)")
print()
for city, label in [(1, "in a city in 1966"), (0, "not in a city in 1966")]:
    close, other = cell(1, city), cell(0, city)
    gap, w = close.schooling.mean() - other.schooling.mean(), weight(len(close), len(other))
    gaps.append(gap); weights.append(w)
    print(f"Men {label}:")
    print(f"  Schooling gap = {gap:.3f} years (from the section \"Does the $1.08 fall when city is held fixed?\")")
    print(f"  Weight = ({len(close):,} x {len(other):,}) / {len(close) + len(other):,} = {w:.1f}")
    print()
near_coefficient = sum(g * w for g, w in zip(gaps, weights)) / sum(weights)
print("Near-college coefficient:")
print(f"  = [({gaps[0]:.3f} x {weights[0]:.1f}) + ({gaps[1]:.3f} x {weights[1]:.1f})] / ({weights[0]:.1f} + {weights[1]:.1f}) = {near_coefficient:.3f} years")

Weight = (men near college x men not near college) / (all men in the group)

Men in a city in 1966:
  Schooling gap = 0.419 years (from the section "Does the $1.08 fall when city is held fixed?")
  Weight = (1,641 x 314) / 1,955 = 263.6

Men not in a city in 1966:
  Schooling gap = 0.718 years (from the section "Does the $1.08 fall when city is held fixed?")
  Weight = (412 x 643) / 1,055 = 251.1

Near-college coefficient:
  = [(0.419 x 263.6) + (0.718 x 251.1)] / (263.6 + 251.1) = 0.565 years


**Step 2, the city coefficient, the second 0.56.** The same calculation with the two columns swapped: the schooling gap is now between men in a city and men not in a city, and the two groups are near-college men and not-near-college men.

In [22]:
gaps, weights = [], []
print("Weight = (men in a city x men not in a city) / (all men in the group)")
print()
for near, label in [(1, "Near-college men"), (0, "Not-near-college men")]:
    in_city, not_city = cell(near, 1), cell(near, 0)
    gap, w = in_city.schooling.mean() - not_city.schooling.mean(), weight(len(in_city), len(not_city))
    gaps.append(gap); weights.append(w)
    print(f"{label}:")
    print(f"  Schooling gap = (mean schooling, in a city) - (mean schooling, not in a city) = {in_city.schooling.mean():.3f} - {not_city.schooling.mean():.3f} = {gap:.3f} years")
    print(f"  Weight = ({len(in_city):,} x {len(not_city):,}) / {len(in_city) + len(not_city):,} = {w:.1f}")
    print()
city_coefficient = sum(g * w for g, w in zip(gaps, weights)) / sum(weights)
print("City coefficient:")
print(f"  = [({gaps[0]:.3f} x {weights[0]:.1f}) + ({gaps[1]:.3f} x {weights[1]:.1f})] / ({weights[0]:.1f} + {weights[1]:.1f}) = {city_coefficient:.3f} years")

Weight = (men in a city x men not in a city) / (all men in the group)



Near-college men:
  Schooling gap = (mean schooling, in a city) - (mean schooling, not in a city) = 13.616 - 13.172 = 0.444 years
  Weight = (1,641 x 412) / 2,053 = 329.3

Not-near-college men:
  Schooling gap = (mean schooling, in a city) - (mean schooling, not in a city) = 13.197 - 12.454 = 0.743 years
  Weight = (314 x 643) / 957 = 211.0

City coefficient:
  = [(0.444 x 329.3) + (0.743 x 211.0)] / (329.3 + 211.0) = 0.561 years


**Step 3, the starting value, the 12.51.** The starting value is predicted schooling when both columns are 0. A least-squares equation with a starting value always passes through the mean of every column. So the mean schooling of all 3,010 men equals the equation at the mean near-college value and the mean city value. Solving that for the starting value gives the formula in the output.

In [23]:
start = card.schooling.mean() - near_coefficient * card.near_college.mean() - city_coefficient * card.city.mean()
print("Starting value = (mean schooling) - (near-college coefficient x share near college) - (city coefficient x share in a city)")
print()
print(f"  Mean schooling, all 3,010 men = {card.schooling.mean():.3f} years")
print(f"  Share near college = {card.near_college.sum():,} / 3,010 = {card.near_college.mean():.3f}")
print(f"  Share in a city = {card.city.sum():,} / 3,010 = {card.city.mean():.3f}")
print()
print(f"  Starting value = {card.schooling.mean():.3f} - ({near_coefficient:.3f} x {card.near_college.mean():.3f}) - ({city_coefficient:.3f} x {card.city.mean():.3f}) = {start:.2f} years")
print()
print("Stage 1 in numbers:")
print(f"  Predicted schooling = {start:.2f} + ({near_coefficient:.2f} x near college) + ({city_coefficient:.2f} x city)")

Starting value = (mean schooling) - (near-college coefficient x share near college) - (city coefficient x share in a city)

  Mean schooling, all 3,010 men = 13.263 years
  Share near college = 2,053 / 3,010 = 0.682
  Share in a city = 1,955 / 3,010 = 0.650

  Starting value = 13.263 - (0.565 x 0.682) - (0.561 x 0.650) = 12.51 years

Stage 1 in numbers:
  Predicted schooling = 12.51 + (0.56 x near college) + (0.56 x city)
